# Data Normalisation

## Why do we need a normaliser?

The project pulls data from multiple sources — Google Calendar and Google Tasks, with Microsoft Graph (Teams Calendar) planned for later. Each API returns data in its own format with different field names, date formats, and status values.

For example:
- A Google Calendar event uses `summary` for the title
- A Google Task uses `title`
- A calendar event has both a `start` and `end` with nested `dateTime` or `date` keys
- A task only has a `due` date, and no concept of a start time

If the scheduling engine had to deal with all these differences, it would become messy and hard to extend. Instead, we normalise everything into one consistent schema early on, so every downstream component (database, AI engine, API) works with the same structure.

---

## The Unified Schema

Every item — whether it's a calendar event or a task — will be represented like this:

| Field | Type | Description |
|---|---|---|
| `id` | string | Unique ID from the source API |
| `title` | string | Name of the event or task |
| `start` | string \| None | ISO 8601 start datetime |
| `end` | string \| None | ISO 8601 end datetime or due date |
| `type` | string | `'event'` or `'task'` |
| `source` | string | Where the data came from (e.g. `'google_calendar'`) |
| `priority` | None | Reserved for the AI engine to fill in later |
| `status` | string | `'confirmed'`, `'pending'`, or `'completed'` |
| `description` | string \| None | Notes or description text |
| `task_list` | string \| None | Task list name (tasks only) |

---

## Step 1 — Understand the raw API responses

Before writing any normalisation code, it helps to see what the raw data actually looks like. Below are representative examples of what the Google Calendar and Google Tasks APIs return.

In [ ]:
# Example raw Google Calendar event (as returned by the API)
raw_event = {
    'id': 'abc123',
    'summary': 'Team Standup',
    'description': 'Daily sync with the team.',
    'start': {'dateTime': '2026-03-30T09:00:00+01:00', 'timeZone': 'Europe/London'},
    'end':   {'dateTime': '2026-03-30T09:30:00+01:00', 'timeZone': 'Europe/London'},
    'status': 'confirmed',
}

# Example raw Google Calendar all-day event (uses 'date' instead of 'dateTime')
raw_all_day_event = {
    'id': 'def456',
    'summary': 'Bank Holiday',
    'start': {'date': '2026-04-01'},
    'end':   {'date': '2026-04-02'},
    'status': 'confirmed',
}

# Example raw Google Task (as returned by the API)
raw_task = {
    'id': 'ghi789',
    'title': 'Write project report',
    'notes': 'Include the Q1 results.',
    'due': '2026-04-05T00:00:00.000Z',
    'status': 'needsAction',
    'task_list_title': 'Work',
}

# Example completed task
raw_completed_task = {
    'id': 'jkl012',
    'title': 'Send invoices',
    'due': None,
    'status': 'completed',
    'task_list_title': 'Admin',
}

print('Raw event keys:    ', list(raw_event.keys()))
print('Raw task keys:     ', list(raw_task.keys()))

Notice the differences:
- Events use `summary`, tasks use `title`
- Events have a nested `start`/`end` dict that can contain either `dateTime` (timed events) or `date` (all-day events)
- Tasks only have a `due` date, no start time
- Task status is `'needsAction'` or `'completed'` — not immediately human-readable

The normaliser's job is to iron out all of these differences.

---

## Step 2 — Normalise a Google Calendar event

The `normalize_event` function extracts only the fields we care about and maps them to the unified schema. Key decisions:
- `summary` → `title` (with a fallback if the field is missing)
- `start.dateTime` is preferred over `start.date` — the `or` handles both cases cleanly
- `priority` is set to `None` because the AI engine will assign this later

In [ ]:
def normalize_event(event: dict) -> dict:
    start = event.get('start', {})
    end = event.get('end', {})

    return {
        'id':          event.get('id'),
        'title':       event.get('summary', 'Untitled Event'),
        'start':       start.get('dateTime') or start.get('date'),
        'end':         end.get('dateTime') or end.get('date'),
        'type':        'event',
        'source':      'google_calendar',
        'priority':    None,
        'status':      event.get('status', 'confirmed'),
        'description': event.get('description'),
    }


# Test with a timed event
print('Timed event:')
print(normalize_event(raw_event))
print()

# Test with an all-day event
print('All-day event:')
print(normalize_event(raw_all_day_event))

Both event types produce the same schema — the `start.get('dateTime') or start.get('date')` pattern handles the difference transparently.

---

## Step 3 — Normalise a Google Task

Tasks are slightly different to events:
- They have no `start` time, so we set it to `None`
- `due` maps to `end` — it represents the deadline
- The raw status `'needsAction'` is converted to the more readable `'pending'`
- We keep `task_list` as an extra field (tasks only) so we know which list it belongs to

In [ ]:
def normalize_task(task: dict) -> dict:
    raw_status = task.get('status', 'needsAction')
    status = 'completed' if raw_status == 'completed' else 'pending'

    return {
        'id':          task.get('id'),
        'title':       task.get('title', 'Untitled Task'),
        'start':       None,
        'end':         task.get('due'),
        'type':        'task',
        'source':      'google_tasks',
        'priority':    None,
        'status':      status,
        'description': task.get('notes'),
        'task_list':   task.get('task_list_title'),
    }


# Test with a pending task
print('Pending task:')
print(normalize_task(raw_task))
print()

# Test with a completed task
print('Completed task:')
print(normalize_task(raw_completed_task))

---

## Step 4 — Combine and sort everything

`normalize_all` takes both raw lists, normalises each item, combines them, and sorts the result chronologically. Items without any date (tasks with no due date) are pushed to the end.

The sort key is a tuple `(date, type)` — this means if two items share the same date, events are sorted before tasks alphabetically.

In [ ]:
def normalize_all(events: list, tasks: list) -> list:
    normalized = []

    for event in events:
        normalized.append(normalize_event(event))

    for task in tasks:
        normalized.append(normalize_task(task))

    # Sort by date — items with no date sort to the end (empty string < any date string)
    normalized.sort(key=lambda x: (x['start'] or x['end'] or '', x['type']))

    return normalized


# Test combining the examples from above
raw_events = [raw_event, raw_all_day_event]
raw_tasks  = [raw_task, raw_completed_task]

schedule = normalize_all(raw_events, raw_tasks)

for item in schedule:
    print(f"[{item['type'].upper()}] {item['title']}")
    print(f"  start={item['start']}  end={item['end']}  status={item['status']}")
    print()

---

## Step 5 — Using the normaliser with real data

The functions above are saved in `src/grayfia/normalizer.py`. In `main.py`, they're used like this:

```python
from src.grayfia.client import Grayfia
from src.grayfia.normalizer import normalize_all

grayfia = Grayfia()
events   = grayfia.get_events()
tasks    = grayfia.get_tasks()
schedule = normalize_all(events, tasks)
```

The result is a clean, sorted list of dicts that every future component — the database, the AI scheduling engine, and the API layer — can rely on without needing to know anything about the Google API response format.

---

## What's next?

Now that we have a consistent schema, the next step is **Phase 2: Database**.

The normalised schema maps directly to a database table, so storing and querying items will be straightforward. We'll also need to extend the normaliser when Microsoft Graph API integration is added — but because everything already flows through `normalize_all`, that will be a small, isolated change.